# FakeRecogna 2.0 — Demo

Tour guiado pelo pipeline de detecção de fake news em português brasileiro.
Esse notebook **roda em alguns minutos** (sem treinar modelos pesados) e
ilustra como usar a API limpa do pacote `fakerecogna2`. Para o experimento
completo (BERTimbau FT, CNN/LSTM/ConvLSTM, ensembles, PLMs maiores), use os
scripts em [`scripts/`](../scripts/) — em particular `python scripts/run_all.py`.

## Estrutura do pacote

- `fakerecogna2.data` — carregamento, integridade, splits
- `fakerecogna2.preprocessing` — limpeza, sumarização, equalizador linguístico
- `fakerecogna2.features` — embeddings BERTimbau, TF-IDF, dataloaders
- `fakerecogna2.models` — baselines, CNN/LSTM, ensembles, BERTimbau FT, PLMs
- `fakerecogna2.evaluation` — métricas, bootstrap, calibração, stress tests
- `fakerecogna2.statistics` — log-odds Dirichlet, McNemar Holm
- `fakerecogna2.explainability` — LIME, IG, Attention Rollout
- `fakerecogna2.adversarial` — perturbações de texto, back-translation
- `fakerecogna2.deployment` — latência, VRAM, Pareto F1×latência
- `fakerecogna2.reports` — relatório consolidado MD + JSON

## 0. Setup

Cria o `ExperimentContext` (dataclass que carrega o estado dinâmico do
experimento — DataFrame, tokenizer, modelos, embeddings, predições) e
fixa as seeds.

In [ ]:
import sys
from pathlib import Path

# Permite importar fakerecogna2 mesmo sem `pip install -e .`
sys.path.insert(0, str(Path.cwd().parent / "src"))

from fakerecogna2 import ExperimentContext
from fakerecogna2.config import ensure_dirs
from fakerecogna2.utils import get_logger, set_global_seeds

ensure_dirs()
ctx = ExperimentContext()
set_global_seeds(ctx.seed)
log = get_logger()
print(f"seed={ctx.seed}  device={ctx.device}  max_seq_len={ctx.max_seq_len}")

## 1. Carregamento do dataset (FakeRecogna 2.0 Abstrativa)

`load_and_prepare` faz tudo em uma chamada: download HF → normalize schema →
encode labels → parse de datas → dedupe exato (SHA-1) e near-duplicate
(MinHash LSH @ 0.85 Jaccard).

In [ ]:
from fakerecogna2.data import load_and_prepare

ctx.df, encoder, class_names = load_and_prepare("abstrativa")
ctx.extras["label_encoder"] = encoder
ctx.extras["class_names"] = class_names

print(f"Shape final: {ctx.df.shape}")
print(f"Classes: {class_names}")
ctx.df[["text", "label", "label_enc", "source"]].head(3)

## 2. EDA lexical — log-odds com Dirichlet prior (Monroe et al. 2008)

Identifica os termos mais característicos de cada classe. Métrica preferida
a TF-IDF ou chi² porque produz estatística *direcional* (positivo → classe
A, negativo → classe B) e não penaliza termos frequentes se eles forem
discriminativos.

Usamos uma **amostra de 3000** pra rodar em segundos. O experimento real
(script `02_preprocess_text.py`) roda no dataset inteiro.

In [ ]:
from fakerecogna2.statistics import lex_analysis

df_sample = ctx.df.sample(min(3000, len(ctx.df)), random_state=ctx.seed)
lex_results = lex_analysis(df_sample, name="demo", top_k=10, save=False)

## 3. Splits estratificados (70/10/20)

`make_random_splits` retorna 6 listas/arrays: `X_train, X_val, X_test,
y_train, y_val, y_test`. As outras estratégias disponíveis são
`make_temporal_splits` (treino no passado, teste no futuro) e
`make_source_splits` (GroupShuffleSplit — fontes em treino e teste não
se sobrepõem).

In [ ]:
from fakerecogna2.data import make_random_splits

X_tr, X_vl, X_te, y_tr, y_vl, y_te = make_random_splits(ctx.df, text_col="text")
print(f"Train: {len(X_tr)}  Val: {len(X_vl)}  Test: {len(X_te)}")
print(f"Distribuição train: {dict(zip(*__import__('numpy').unique(y_tr, return_counts=True)))}")

## 4. Baseline rápido: TF-IDF + Logistic Regression

Pra dar uma noção do *floor* de desempenho. Sem GPU, treina em segundos.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

from fakerecogna2.features import fit_tfidf

tfidf = fit_tfidf(X_tr)
Xtr_v, Xte_v = tfidf.transform(X_tr), tfidf.transform(X_te)

clf = LogisticRegression(max_iter=1000, random_state=ctx.seed).fit(Xtr_v, y_tr)
y_pred = clf.predict(Xte_v)
print(classification_report(y_te, y_pred, target_names=class_names, digits=4))

## 5. Bootstrap CI no F1

Mostra que o resultado acima vem com incerteza estatística. 10 000
resamples com reposição → intervalo percentil [2.5%, 97.5%].

In [ ]:
from fakerecogna2.evaluation import bootstrap_ci, f1_macro

f1 = f1_macro(y_te, y_pred)
f1_lo, f1_hi = bootstrap_ci(y_te, y_pred, f1_macro, n_boot=2000)
print(f"F1 macro: {f1*100:.2f}%  [95% CI: {f1_lo*100:.2f}%, {f1_hi*100:.2f}%]")

## 6. Resultados completos (do último `run_all.py`)

Esta célula só funciona se você já rodou `python scripts/run_all.py`
(ou pelo menos `04_train_deep_models.py` + `05_evaluate_models.py`).
Lê os CSVs em `outputs/metrics/` e mostra a tabela consolidada.

In [ ]:
import pandas as pd
from fakerecogna2.config import ARTIFACTS_DIR

final = ARTIFACTS_DIR / "metrics" / "16_final_results.csv"
if final.exists():
    df_final = pd.read_csv(final).sort_values("F1", ascending=False)
    display(df_final.round(4))
else:
    print("Tabela ainda não gerada — rode `python scripts/run_all.py` (ou ao menos 04+05).")

## 7. Como reproduzir o experimento completo

```powershell
# Setup (uma vez)
python -m venv .venv
.venv\Scripts\Activate.ps1
pip install -r requirements.txt
python -m spacy download pt_core_news_sm

# Verificar ambiente
python scripts/00_validate_environment.py

# Pipeline completo (2–4h em RTX 5080)
python scripts/run_all.py

# Subset rápido (sem cross-dataset, PLMs maiores, adversarial)
python scripts/run_all.py --quick

# Uma etapa só
python scripts/04_train_deep_models.py --no-bert --epochs 5
```

Resultados saem em `outputs/`:
- `outputs/metrics/*.csv` — todas as tabelas
- `outputs/figures/*.png` — todas as figuras
- `outputs/relatorio_final.md` — narrativa consolidada
- `outputs/resultados.json` — números estruturados

Veja [`README.md`](../README.md) e [`CLAUDE.md`](../CLAUDE.md) para mais detalhes.